# PBG de Mendoza (1970-2022) — dos tramos NO empalmados

La DEIE publica el PBG en tramos con distinto año base. Entre **1986-1993** y **1991-2003** hay años de solapamiento (1991-1993), así que ese empalme tiene respaldo empírico. Entre **1970-1985** y **1986-1993** NO hay solapamiento: se probó usar la tabla oficial de conversión monetaria (Austral→Peso) junto con el IPC, y el resultado da un salto de nivel de ~30.000 veces respecto al tramo siguiente — señal de que el cambio de año base implica un recálculo metodológico completo, no solo un cambio de moneda y precios.

Por eso, siguiendo la recomendación de la propia DEIE (evitar comparar valores absolutos del PBG entre años base distintos), **no se fuerza una continuidad**:
- **1970-1985**: índice propio (1970=100), sin convertir a pesos.
- **1986-2022**: empalmado (con solapamiento real) y reexpresado a precios constantes de 1999.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## 1. Datos crudos por tramo (fuente: DEIE Mendoza + IPC INDEC)

In [ ]:
# Tramo A: 1970-1985, a valores constantes de 1970, en Australes
PBG_A = {1970: 313.43, 1971: 334.41, 1972: 356.26, 1973: 364.33, 1974: 406.37, 1975: 355.2, 1976: 404.05, 1977: 380.63, 1978: 423.6, 1979: 481.05, 1980: 429.24, 1981: 351.83, 1982: 360.43, 1983: 419.26, 1984: 404.44, 1985: 384.61}

# Tramo B: 1986-1993, en pesos de 1996 (año base 1996)
PBG_B = {1986: 267608.7, 1987: 271052.0, 1988: 264473.0, 1989: 253410.0, 1990: 253627.0, 1991: 259579.0, 1992: 277469.0, 1993: 296965.0}

# Tramo C: 1991-2003, en miles de pesos de 1993
PBG_C = {1991: 6485506.946923, 1992: 7035025.586794, 1993: 7761508.224701, 1994: 8098028.310572, 1995: 7908811.385646, 1996: 8135361.877227, 1997: 8905959.444905, 1998: 9488295.932283, 1999: 9252615.169366, 2000: 9002038.451338, 2001: 8322993.49667, 2002: 7772198.039427, 2003: 9339929.594535}

# Tramo D: 2004-2022, en miles de pesos de 1993
PBG_D = {2004: 10874929.097084, 2005: 11380763.411128, 2006: 12259303.976934, 2007: 12770718.617713, 2008: 13065173.973136, 2009: 12700761.719855, 2010: 13262431.723353, 2011: 13739583.70382, 2012: 13545085.639385, 2013: 14211513.620518, 2014: 13687924.078648, 2015: 14189502.676874, 2016: 13373002.759949, 2017: 13655492.409271, 2018: 13585607.412423, 2019: 13410851.445972, 2020: 12342645.880974, 2021: 13635887.646933, 2022: 14178694.442282}

# IPC INDEC, nivel general, base 1999=100 (1970-2021)
IPC = {1970: 4.75e-10, 1971: 6.4e-10, 1972: 1.015e-09, 1973: 1.627e-09, 1974: 2.021e-09, 1975: 5.714e-09, 1976: 3.1086e-08, 1977: 8.5808e-08, 1978: 2.36409e-07, 1979: 6.13507e-07, 1980: 1.231702e-06, 1981: 2.51855e-06, 1982: 6.668583e-06, 1983: 2.9595142e-05, 1984: 0.000215077833, 1985: 0.0016607925, 1986: 0.003157004167, 1987: 0.007303241667, 1988: 0.03235, 1989: 1.028553583333, 1990: 24.8289, 1991: 67.4531, 1992: 84.2489, 1993: 93.189, 1994: 97.0818, 1995: 100.3594, 1996: 100.5156, 1997: 101.0469, 1998: 101.9813, 1999: 100.7915, 2000: 99.8449, 2001: 98.781666666667, 2002: 124.335, 2003: 141.05, 2004: 147.28, 2005: 167.48, 2006: 179.08, 2007: 194.89, 2008: 238.54536, 2009: 287.92424952, 2010: 334.56797794224, 2011: 427.912443788125, 2012: 527.616043190758, 2013: 650.550581254205, 2014: 844.414654467958, 2015: 1192.313492108756, 2016: 1513.045821486012, 2017: 2066.820592149892, 2018: 2546.322969528668, 2019: 3860.22562180546, 2020: 5921.586103849576, 2021: 8136.259306689318}

## 2. Tramo 1970-1985: índice propio (1970=100), sin empalmar

In [ ]:
indice_A = {y: v / PBG_A[1970] * 100 for y, v in PBG_A.items()}
df_A = pd.DataFrame(sorted(indice_A.items()), columns=["anio", "indice_1970_100"])
df_A

## 3. Tramo 1986-2022: empalme B→C (con solapamiento real 1991-1993) y precios de 1999

In [ ]:
coef_B = PBG_C[1993] / PBG_B[1993]
print(f"Coeficiente de empalme B->C (ancla 1993): {coef_B:.4f}")

factor_1999 = IPC[1999] / IPC[1993]
print(f"Factor de reexpresion a precios de 1999: {factor_1999:.4f}")

serie_1993 = {}
for y, v in PBG_B.items():
    if y <= 1990:
        serie_1993[y] = v * coef_B
for y, v in PBG_C.items():
    serie_1993[y] = v
for y, v in PBG_D.items():
    serie_1993[y] = v

serie_1999 = {y: v * factor_1999 for y, v in serie_1993.items()}

df_B = (pd.DataFrame(sorted(serie_1999.items()), columns=["anio", "pbg_miles_1999"])
          .assign(var_pct=lambda d: d["pbg_miles_1999"].pct_change() * 100))
df_B

In [ ]:
df_A.to_csv("pbg_mendoza_indice_1970_1985.csv", index=False)
df_B.to_csv("pbg_mendoza_precios_1999_1986_2022.csv", index=False)

## 4. Gráfico — dos ejes Y, con quiebre visual en 1985/1986

In [ ]:
navy, rust, teal, lightgrey = "#1B2A4A", "#C0392B", "#1F8A8C", "#D9D9D9"

fig, ax1 = plt.subplots(figsize=(11.5, 6), dpi=150)
ax2 = ax1.twinx()

ax1.plot(df_A["anio"], df_A["indice_1970_100"], color=rust, linewidth=2.2)
ax2.plot(df_B["anio"], df_B["pbg_miles_1999"], color=navy, linewidth=2.2)

ax1.axvspan(1985, 1986, color=teal, alpha=0.08)
ax1.axvline(1985.5, color="#888", linestyle="--", linewidth=1)

ax1.set_title("Evolución del PBG de Mendoza (1970-2022)\ndos tramos NO comparables en nivel (distinto año base)",
              fontsize=13.5, fontweight="bold", color=navy, loc="left")
ax1.set_ylabel("Índice 1970-1985 (1970=100)", fontsize=10, color=rust)
ax1.tick_params(axis="y", colors=rust, labelsize=9)
ax1.set_xlim(1969, 2023)

ax2.set_ylabel("Miles de $ ctes. de 1999 — tramo 1986-2022", fontsize=10, color=navy)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:,.0f}M"))
ax2.tick_params(axis="y", colors=navy, labelsize=9)

ax1.set_xlabel("Año", fontsize=10, color="#333333")
ax1.grid(axis="y", color=lightgrey, linewidth=0.6)
ax1.spines[["top"]].set_visible(False)
ax2.spines[["top"]].set_visible(False)
ax1.tick_params(axis="x", colors="#555555", labelsize=9)

fig.text(0.01, -0.03,
         "Fuente: elaboración propia en base a DEIE Mendoza (PBG 1970-2022) e IPC INDEC (base 1999=100).\n"
         "1970-1985: índice propio, sin convertir a pesos. 1986-2022: empalme con solapamiento real, a precios de 1999.",
         fontsize=7, color="#777777")

plt.tight_layout()
plt.savefig("pbg_mendoza_1999.png", bbox_inches="tight")
plt.show()

## 5. Descargar resultados (Colab)

In [ ]:
from google.colab import files
files.download("pbg_mendoza_indice_1970_1985.csv")
files.download("pbg_mendoza_precios_1999_1986_2022.csv")
files.download("pbg_mendoza_1999.png")